In [2]:
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os
import torch
import networkx as nx
import torch.linalg as LA

import math
from typing import Optional, Tuple, List, Union, Dict
import seaborn as sns

### Loading plots

In [3]:
# Loading file
file_path = "/mnt/lts4/scratch/home/carballo/MechInt/DeFoG/outputs/2025-07-16/14-53-50-sbm-sbm/attention/attn_maps_0_4.pt"

# Load the data with pickle
with open(file_path, "rb") as f:
    data = pickle.load(f)

/mnt/lts4/scratch/home/carballo/MechInt/DeFoG/.pixi/envs/default/lib/python3.11/site-packages/torch/storage.py:414: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torc

In [4]:
# data = data.squeeze(1)   # shape: (L, b = 1, N, N, h) -> (L, N, N, h)
data.shape
# Format should be (b, L, h, N, N)
data = data.permute(1, 0, 4, 2, 3)  # shape: (b = 1, L, N, N, h) -> (b = 1, L, h, N, N)
data.shape

torch.Size([1, 8, 8, 63, 63])

### Raw and averaged maps

In [7]:
out_dir = "../maps/raw"
os.makedirs(out_dir, exist_ok=True)

In [8]:
def plot_attention_maps(
    data: Union[torch.Tensor, np.ndarray],
    out_dir: str,
    mode: str = "grid",
    share_clim: bool = True,
    dpi: int = 150,
    cmap: str = "viridis",
    max_cols: int = 4,
    show_axis: bool = False,
    node_mask: Optional[np.ndarray] = None,
) -> List[str]:
    """
    Plot attention maps from tensor with shape (B, L, H, N, N).

    Args:
        data: torch.Tensor or np.ndarray of shape (B, L, H, N, N).
        out_dir: directory where images will be saved. Subfolders per-batch will be created.
        mode: 'grid' (one figure per (batch,layer) with all heads) or 'single' (one file per head).
        share_clim: if True, use same vmin/vmax across heads within same (batch,layer).
        dpi: output dpi for saved images.
        cmap: colormap for heatmaps.
        max_cols: maximum columns in grid when mode='grid'.
        show_axis: if True, show axis ticks; otherwise hide (like your original).
        node_mask: optional boolean array of length N; masked nodes are grayed out (rows/cols).
    Returns:
        List of saved filenames.
    """

    # Convert to numpy
    if torch.is_tensor(data):
        data = data.detach().cpu().numpy()
    data = np.asarray(data)
    if data.ndim != 5:
        raise ValueError("data must have shape (B, L, H, N, N)")

    B, L, H, N1, N2 = data.shape
    assert N1 == N2, "attention maps must be square (N x N)"
    os.makedirs(out_dir, exist_ok=True)
    saved_files = []

    # Prepare node mask if provided
    if node_mask is not None:
        node_mask = np.asarray(node_mask).astype(bool)
        if node_mask.shape[0] != N1:
            raise ValueError("node_mask must have same length as N")
        valid_mask = np.outer(node_mask, node_mask).astype(float)
    else:
        valid_mask = np.ones((N1, N2), dtype=float)

    for b in range(B):
        batch_dir = os.path.join(out_dir, f"batch_{b}")
        os.makedirs(batch_dir, exist_ok=True)

        for l in range(L):
            # shape (H, N, N)
            heads = data[b, l, :, :, :]  
            heads_masked = heads * valid_mask

            # shared color scale across heads
            if share_clim:
                vmin = float(heads_masked.min())
                vmax = float(heads_masked.max())
            else:
                vmin = vmax = None

            if mode == "single":
                for h in range(H):
                    fig, ax = plt.subplots(figsize=(5, 4))
                    sns.heatmap(
                        heads_masked[h],
                        cmap=cmap,
                        cbar=True,
                        vmin=vmin,
                        vmax=vmax,
                        ax=ax,
                    )
                    ax.set_title(f"Batch {b}, Layer {l+1}, Head {h+1}")
                    if not show_axis:
                        ax.axis("off")

                    # Gray overlay for masked nodes
                    if node_mask is not None:
                        invalid_idx = np.where(~node_mask)[0]
                        for idx in invalid_idx:
                            ax.axhspan(idx, idx + 1, color="lightgray", alpha=0.5)
                            ax.axvspan(idx, idx + 1, color="lightgray", alpha=0.5)

                    filename = os.path.join(batch_dir, f"layer{l+1}_head{h+1}.png")
                    fig.savefig(filename, bbox_inches="tight", dpi=dpi)
                    plt.close(fig)
                    saved_files.append(filename)

            elif mode == "grid":
                ncols = min(max_cols, H)
                nrows = math.ceil(H / ncols)
                fig, axes = plt.subplots(
                    nrows, ncols, figsize=(4 * ncols, 3 * max(1, nrows)), squeeze=False
                )
                axes_flat = axes.flatten()

                for idx in range(nrows * ncols):
                    ax = axes_flat[idx]
                    if idx < H:
                        sns.heatmap(
                            heads_masked[idx],
                            cmap=cmap,
                            cbar=False,
                            vmin=vmin,
                            vmax=vmax,
                            ax=ax,
                        )
                        ax.set_title(f"H{idx+1}")
                        if not show_axis:
                            ax.axis("off")
                        if node_mask is not None:
                            invalid_idx = np.where(~node_mask)[0]
                            for jdx in invalid_idx:
                                ax.axhspan(jdx, jdx + 1, color="lightgray", alpha=0.5)
                                ax.axvspan(jdx, jdx + 1, color="lightgray", alpha=0.5)
                    else:
                        ax.axis("off")

                cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
                mpl_im = axes_flat[0].collections[0]
                fig.colorbar(mpl_im, cax=cbar_ax)
                fig.suptitle(f"Batch {b}, Layer {l+1} (H={H})", fontsize=14)

                filename = os.path.join(batch_dir, f"layer{l+1}_all_heads_grid.png")
                fig.savefig(filename, bbox_inches="tight", dpi=dpi)
                plt.close(fig)
                saved_files.append(filename)

            else:
                raise ValueError("mode must be 'grid' or 'single'")

            # Also save mean across heads
            mean_attn = heads_masked.mean(axis=0)
            fig, ax = plt.subplots(figsize=(5, 4))
            sns.heatmap(mean_attn, cmap=cmap, cbar=True, vmin=vmin, vmax=vmax, ax=ax)
            ax.set_title(f"Batch {b}, Layer {l+1}, Heads mean")
            if not show_axis:
                ax.axis("off")
            if node_mask is not None:
                invalid_idx = np.where(~node_mask)[0]
                for idx in invalid_idx:
                    ax.axhspan(idx, idx + 1, color="lightgray", alpha=0.5)
                    ax.axvspan(idx, idx + 1, color="lightgray", alpha=0.5)
            filename = os.path.join(batch_dir, f"layer{l+1}_heads_mean.png")
            fig.savefig(filename, bbox_inches="tight", dpi=dpi)
            plt.close(fig)
            saved_files.append(filename)

    return saved_files

In [ ]:
# suppose `data` is shape (L, B, N, N, h)
saved = plot_attention_maps(data, out_dir="../../maps/raw", mode="grid", share_clim=True)
print("Wrote", len(saved), "files.")

# for layer in range(layers):
#     for head in range(heads):
#         attn_map = data[layer, :, :, head].detach().numpy()
        
#         plt.figure(figsize=(5, 4))
#         sns.heatmap(attn_map, cmap="viridis", cbar=True)
#         plt.title(f"Layer {layer+1}, Head {head+1}")
#         plt.axis("off")
        
#         filename = os.path.join(out_dir, f"layer{layer+1}_head{head+1}.png")
#         plt.savefig(filename, bbox_inches="tight", dpi=150)
#         plt.close()

Wrote 16 files.


Mean over heads per layer (old)

In [18]:
out_dir = "../maps/mean"
os.makedirs(out_dir, exist_ok=True)

In [ ]:
# Average over heads and plot the heatmap
# mean_data = data.mean(dim=-1)  # shape: (L, N, N, h) -> (L, N, N)

# for layer in range(layers):
#     plt.figure(figsize=(10, 8))
#     sns.heatmap(mean_data[layer], cmap="viridis", cbar=True)
#     plt.title(f"Layer {layer + 1} Attention Map")
#     plt.savefig(f"{out_dir}/layer_{layer + 1}_mean.png")
#     plt.close()

### Attention rollout / Attention flow

From Quantifying Attention Flow in Transformers (Abnar & Zuidema, 2020)

Attention rollout (old)

In [ ]:
out_dir = "../../maps/rollout"
os.makedirs(out_dir, exist_ok=True)

In [12]:
def plot_attention_rollout(
    attn: torch.Tensor,
    out_dir: str,
    residual: bool = True,
    max_nodes_to_plot: int = 6,
):
    """
    Compute and visualize graph attention rollout per batch.

    Args:
        attn: Tensor (B, L, H, N, N)
        out_dir: directory to save plots
        residual: whether to include residual (0.5*A + 0.5*I)
        max_nodes_to_plot: how many source nodes to visualize per batch

    Returns:
        rollout: Tensor (B, L, N, N) cumulative attention per layer
    """

    os.makedirs(out_dir, exist_ok=True)

    if attn.ndim != 5:
        raise ValueError("attn must have shape (B, L, H, N, N)")

    B, L, H, N, N2 = attn.shape
    assert N == N2, "attention maps must be square (N x N)"

    # Average over heads
    attn = attn.mean(dim=2)  # (B, L, N, N)

    I = torch.eye(N, device=attn.device).unsqueeze(0)
    rollout = torch.zeros((B, L, N, N), device=attn.device)

    for b in range(B):
        result = torch.eye(N, device=attn.device)
        for l in range(L):
            A = attn[b, l]
            if residual:
                A = 0.5 * (A + I[0])
            A = A / (A.sum(dim=-1, keepdim=True) + 1e-8)
            result = A @ result
            rollout[b, l] = result

    # Visualization
    node_indices = range(min(N, max_nodes_to_plot))

    for b in range(B):
        batch_dir = os.path.join(out_dir, f"batch_{b}")
        os.makedirs(batch_dir, exist_ok=True)

        # Plot rollout evolution per source node
        for node in node_indices:
            plt.figure(figsize=(6, 4))
            sns.heatmap(
                rollout[b, :, node, :].detach().cpu().numpy(),
                cmap="viridis",
                cbar=True,
            )
            plt.xlabel("Target node index")
            plt.ylabel("Layer (from input to top)")
            plt.title(f"Batch {b}: Node {node} rollout attention")
            plt.tight_layout()
            plt.savefig(os.path.join(batch_dir, f"node{node}_rollout.png"), dpi=150)
            plt.close()

        # Final cumulative rollout matrix
        plt.figure(figsize=(5, 4))
        sns.heatmap(
            rollout[b, -1].detach().cpu().numpy(),
            cmap="magma",
            cbar=True,
        )
        plt.title(f"Batch {b}: Final rollout attention")
        plt.xlabel("Source node")
        plt.ylabel("Target node")
        plt.tight_layout()
        plt.savefig(os.path.join(batch_dir, "rollout_final.png"), dpi=150)
        plt.close()

    return rollout

In [17]:
def plot_attention_flow(
    attn: torch.Tensor,
    out_dir: str,
    residual: bool = True,
    threshold: float = 1e-4,
    max_nodes_to_plot: int = 6,
    per_layer_plot: bool = False,
):
    """
    Compute and visualize graph attention flow per batch using max-flow formulation.
    Handles attention tensors with shape (B, L, H, N, N).

    Args:
        attn: torch.Tensor, shape (B, L, H, N, N)
        out_dir: directory where plots are saved
        residual: whether to include residual connections (0.5*A + 0.5*I)
        threshold: minimum attention weight to consider as edge
        max_nodes_to_plot: number of source nodes to visualize
        per_layer_plot: if True, visualize flow evolution per layer per node

    Returns:
        flows: torch.Tensor (B, N, N) — final flow matrix per batch
    """

    os.makedirs(out_dir, exist_ok=True)

    if attn.ndim != 5:
        raise ValueError("attn must have shape (B, L, H, N, N)")

    B, L, H, N, N2 = attn.shape
    assert N == N2, "attention maps must be square (N x N)"

    # Average heads
    attn = attn.mean(dim=2)  # (B, L, N, N)
    flows = torch.zeros((B, N, N))
    I = np.eye(N)

    for b in range(B):
        batch_dir = os.path.join(out_dir, f"batch_{b}")
        os.makedirs(batch_dir, exist_ok=True)

        # --- Build layered DAG ---
        G = nx.DiGraph()
        for l in range(L):
            for n in range(N):
                G.add_node((l, n))

        # --- Add directed edges with capacities ---
        for l in range(1, L):
            A = attn[b, l].detach().cpu().numpy()
            if residual:
                A = 0.5 * (A + I)
            A = A / (A.sum(axis=-1, keepdims=True) + 1e-8)
            A[A < threshold] = 0

            for i in range(N):
                targets = np.where(A[i] > 0)[0]
                for j in targets:
                    # Edge from (layer l, node i) to (layer l-1, node j)
                    G.add_edge((l, i), (l - 1, j), capacity=float(A[i, j]))

        # --- Optional per-layer flow visualization ---
        if per_layer_plot:
            node_indices = range(min(N, max_nodes_to_plot))
            layer_flows = torch.zeros((L, N, N))
            for l in range(L):
                for src in node_indices:
                    for tgt in range(N):
                        try:
                            val = nx.maximum_flow_value(G, (l, src), (0, tgt), capacity="capacity")
                        except nx.NetworkXError:
                            val = 0.0
                        layer_flows[l, tgt, src] = val

            # Plot per-layer flows for each source node
            for node in node_indices:
                plt.figure(figsize=(6, 4))
                sns.heatmap(
                    layer_flows[:, :, node].numpy(),
                    cmap="viridis",
                    cbar=True,
                )
                plt.xlabel("Target node index")
                plt.ylabel("Layer (from input to top)")
                plt.title(f"Batch {b}: Node {node} flow per layer")
                plt.tight_layout()
                plt.savefig(os.path.join(batch_dir, f"node{node}_flow_per_layer.png"), dpi=150)
                plt.close()

        # --- Compute final flow matrix (from last to first layer) ---
        for src in range(N):
            for tgt in range(N):
                try:
                    val = nx.maximum_flow_value(G, (L - 1, src), (0, tgt), capacity="capacity")
                except nx.NetworkXError:
                    val = 0.0
                flows[b, tgt, src] = val

        # Normalize rows to make comparable across batches
        flows[b] /= (flows[b].sum(dim=-1, keepdim=True) + 1e-8)

        # --- Final flow visualization ---
        plt.figure(figsize=(5, 4))
        sns.heatmap(flows[b].numpy(), cmap="coolwarm", cbar=True)
        plt.title(f"Batch {b}: Final Attention Flow")
        plt.xlabel("Source node")
        plt.ylabel("Target node")
        plt.tight_layout()
        plt.savefig(os.path.join(batch_dir, "flow_final.png"), dpi=150)
        plt.close()

    return flows


In [ ]:
rollout = plot_attention_rollout(data, out_dir="../../maps/rollout", residual=True)
# flow = plot_attention_flow(data, out_dir="../maps/flow", residual=True, per_layer_plot=True) takes too much time
flow = plot_attention_flow(data, out_dir="../../maps/flow", residual=True, per_layer_plot=False)

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7565b9d44690>>
Traceback (most recent call last):
  File "/mnt/lts4/scratch/home/carballo/MechInt/DeFoG/.pixi/envs/default/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


In [ ]:
def fast_attention_flow(
    attn: torch.Tensor, out_dir: str, residual: bool = True,
    max_nodes_to_plot: int = 6, per_layer_plot: bool = False
):
    """
    Fast approximate attention flow for graph transformers.
    
    Args:
        attn: (L, B, N, N, H) attention tensor
        out_dir: folder to save plots
        residual: include residual connection
        max_nodes_to_plot: how many source nodes to visualize
        per_layer_plot: whether to visualize per-layer propagation
    Returns:
        flows: (B, N, N) final approximate attention flow
    """
    os.makedirs(out_dir, exist_ok=True)
    L, B, N, _, H = attn.shape
    attn = attn.mean(dim=-1)  # average heads

    I = torch.eye(N, device=attn.device)
    flows = torch.zeros((B, N, N), device=attn.device)

    for b in range(B):
        # Initialize cumulative min-propagation with identity
        result = I.clone()
        per_layer_results = []

        for l in range(L):
            A = attn[l, b]
            if residual:
                A = 0.5 * (A + I)
            A = A / (A.sum(dim=-1, keepdim=True) + 1e-8)

            # Approximate flow: element-wise min along paths
            # Equivalent to max-flow approximation
            result = torch.min(A.unsqueeze(-1), result.unsqueeze(0)).max(dim=1)[0]
            per_layer_results.append(result.clone())

        # Final flow
        flows[b] = result

        # Per-layer plots
        node_indices = range(min(N, max_nodes_to_plot))
        if per_layer_plot:
            per_layer_tensor = torch.stack(per_layer_results, dim=0)  # (L, N, N)
            for node in node_indices:
                plt.figure(figsize=(6, 4))
                sns.heatmap(
                    per_layer_tensor[:, :, node].detach().cpu().numpy(),
                    cmap="viridis",
                    cbar=True,
                )
                plt.xlabel("Target node index")
                plt.ylabel("Layer (from input to top)")
                plt.title(f"Batch {b}: Node {node} approx flow per layer")
                plt.tight_layout()
                plt.savefig(os.path.join(out_dir, f"batch{b}_node{node}_flow_per_layer.png"), dpi=150)
                plt.close()

        # Final flow plot
        plt.figure(figsize=(5, 4))
        sns.heatmap(flows[b].detach().cpu().numpy(), cmap="coolwarm", cbar=True)
        plt.title(f"Batch {b}: Final approximate flow")
        plt.xlabel("Source node")
        plt.ylabel("Target node")
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f"batch{b}_flow_final.png"), dpi=150)
        plt.close()

    return flows

In [ ]:
# residual_weight = 0.5 # as default in the paper

# alpha = residual_weight
# beta = 1.0 - alpha

# mean_data = data.mean(dim=-1)  # shape: (L, N, N, h) -> (L, N, N)

# # 2. Add residual connection + renormalize
# I = torch.eye(mean_data.size(-1), device=mean_data.device) 
# attn_aug = alpha * mean_data + beta * I
# attn_aug = attn_aug / (attn_aug.sum(dim=-1, keepdim=True) + 1e-12)  # normalize rows

# # 3. Recursive multiplication (rollout)
# rollout = [attn_aug[0]]
# for layer in range(1, layers):
#     rollout.append(rollout[layer - 1] @ attn_aug[layer]) #torch.bmm(attn_aug[layer], rollout)  # (B, N, N)

# # Plot heatmaps
# for layer in range(layers):
#     plt.figure(figsize=(10, 8))
#     sns.heatmap(rollout[layer].detach().cpu().numpy(), cmap="viridis")
#     plt.title(f"Layer {layer + 1} Attention Rollout")
#     plt.savefig(f"{out_dir}/layer_{layer + 1}_rollout.png")
#     plt.close()

Attention Flow (old)

In [32]:
out_dir = "../maps/flow"
os.makedirs(out_dir, exist_ok=True)

In [ ]:
# mean_data = data.mean(dim=-1)  # shape: (L, N, N, h) -> (L, N, N)

# source_layer = layers - 1  # last layer
# source_pos = 0            # first node

# flows = torch.zeros(n_nodes, device=mean_data.device)

# # for b in range(B):
# G = nx.DiGraph()

# # Add edges layer by layer (directed downward)
# for l in range(layers):
#     if l == 0:
#         continue
#     A = mean_data[l]
#     for i in range(n_nodes):
#         for j in range(n_nodes):
#             cap = float(A[i, j].item())
#             if cap > 1e-12:
#                 G.add_edge((l, i), (l - 1, j), capacity=cap)

# # Connect all inputs (layer 0) to a super sink
# for j in range(n_nodes):
#     G.add_edge((0, j), "SINK", capacity=float('inf'))

# # Compute flow from source node to sink
# src = (source_layer, source_pos)
# flow_value, flow_dict = nx.maximum_flow(G, src, "SINK")

# # Extract per-input flows
# for j in range(n_nodes):
#     f = flow_dict.get((0, j), {}).get("SINK", 0.0)
#     flows[j] = f

### Attention graphs

In [5]:
out_dir = "../maps/attngraphs"
os.makedirs(out_dir, exist_ok=True)

Utilities

In [6]:
def _to_numpy(attn: Union[torch.Tensor, np.ndarray]) -> np.ndarray:
    if torch.is_tensor(attn):
        attn = attn.detach().cpu().numpy()
    return np.asarray(attn, dtype=float)

def _normalize_rows(mat: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    s = mat.sum(axis=-1, keepdims=True)
    s[s == 0] = 1.0
    return mat / (s + eps)

def _ensure_layout(attn: np.ndarray) -> Tuple[np.ndarray, str]:
    """
    Accept either (L,B,N,N,H) or (B,L,H,N,N).
    Returns attn in canonical (L, B, N, N, H) format and a string describing original.
    """
    if attn.ndim == 5:
        # ambiguous: could be (L,B,N,N,H) or (B,L,H,N,N)
        L, B, N1, N2, H = attn.shape
        if N1 == N2:
            # assume (L,B,N,N,H)
            return attn, "(L,B,N,N,H)"
    if attn.ndim == 5:
        # try other interpretation: (B,L,H,N,N)
        B, L, H, N1, N2 = attn.shape
        if N1 == N2:
            attn = attn.transpose(1, 0, 3, 4, 2)  # -> (L,B,N,N,H)
            return attn, "(B,L,H,N,N)--converted"
    raise ValueError("attn must be 5D either (L,B,N,N,H) or (B,L,H,N,N)")

Building attention graphs

In [7]:
def build_attention_graphs(
    attn: Union[torch.Tensor, np.ndarray],
    agg_heads: str = "mean",
    agg_layers: str = "multiply",
    residual: bool = True,
    top_k: Optional[int] = None,
    threshold: Optional[float] = None,
    node_mask: Optional[np.ndarray] = None,
    return_nx: bool = True,
) -> Tuple[np.ndarray, Optional[List[nx.DiGraph]]]:
    """
    Build per-batch Attention Graph aggregate matrices and optional NetworkX graphs.

    Args:
        attn: attention tensor in (L,B,N,N,H) (preferred) or (B,L,H,N,N).
        agg_heads: how to aggregate heads -> 'mean' | 'max' | 'median' | 'none' (if none return per-head shape)
        agg_layers: how to aggregate layers -> 'multiply' (matrix product), 'rollout' (cumprod-style with residual),
                    'mean', 'sum'
        residual: whether to include identity before normalization (matches paper: A <- normalize(A + I))
        top_k: if provided, keep top_k outgoing edges per row (speed + sparsity)
        threshold: if provided, zero edges with weight < threshold
        node_mask: boolean array length N; False entries are treated as padded nodes and zeroed out
        return_nx: if True also return per-batch networkx DiGraph objects with 'weight' attributes

    Returns:
        AAgg_per_batch: np.ndarray shape (B, N, N) aggregate attention graphs per batch
        graphs: list of length B with networkx.DiGraph objects (or None if return_nx=False)
    """
    attn = _to_numpy(attn)
    attn, layout = _ensure_layout(attn)  # (L,B,N,N,H)
    L, B, N, N2, H = attn.shape
    assert N == N2

    # Optional mask as float matrix for elementwise multiply
    if node_mask is not None:
        node_mask = np.asarray(node_mask).astype(bool)
        if node_mask.shape[0] != N:
            raise ValueError("node_mask length must equal N")
        mask2d = np.outer(node_mask, node_mask).astype(float)
    else:
        mask2d = np.ones((N, N), dtype=float)

    # Aggregate heads
    if agg_heads == "mean":
        Attn_layer = attn.mean(axis=-1)           # (L, B, N, N)
    elif agg_heads == "max":
        Attn_layer = attn.max(axis=-1)            # (L, B, N, N)
    elif agg_heads == "median":
        Attn_layer = np.median(attn, axis=-1)
    elif agg_heads == "none":
        raise ValueError("agg_heads='none' not supported for AAgg (would need extra dim)")
    else:
        raise ValueError(f"Unknown agg_heads={agg_heads}")

    # apply node mask by zeroing rows/cols for padded nodes
    for l in range(L):
        for b in range(B):
            Attn_layer[l, b] = Attn_layer[l, b] * mask2d

    # optionally add residual and normalize each row
    A_proc = np.zeros_like(Attn_layer)
    I = np.eye(N, dtype=float)
    for l in range(L):
        for b in range(B):
            W = Attn_layer[l, b].astype(float)
            # row-normalize first to have sum 1
            W = _normalize_rows(W)
            if residual:
                A = W + I
                A = _normalize_rows(A)  # matches paper: (0.5 W + 0.5 I) if W was normalized
            else:
                A = W
            # threshold / top-k pruning
            if threshold is not None:
                A[A < threshold] = 0.0
            if top_k is not None and top_k < N:
                mask = np.zeros_like(A, dtype=bool)
                for i in range(N):
                    k = min(top_k, N)
                    idx = np.argpartition(-A[i], k-1)[:k]
                    mask[i, idx] = True
                A = A * mask.astype(float)
                A = _normalize_rows(A)
            A_proc[l, b] = A

    # Aggregate across layers
    AAgg_per_batch = np.zeros((B, N, N), dtype=float)

    if agg_layers == "multiply":
        # Product: R = A_L @ A_{L-1} @ ... @ A_1 @ I  (we follow result = A_l @ result)
        for b in range(B):
            result = np.eye(N, dtype=float)
            for l in range(L):
                result = A_proc[l, b] @ result
            AAgg_per_batch[b] = result
    elif agg_layers == "rollout":
        # rollout: accumulate results per layer (like Abnar & Zuidema) with residual already included
        for b in range(B):
            result = np.eye(N, dtype=float)
            for l in range(L):
                result = A_proc[l, b] @ result
            AAgg_per_batch[b] = result
    elif agg_layers == "mean":
        AAgg_per_batch = A_proc.mean(axis=0)   # (B,N,N)
    elif agg_layers == "sum":
        AAgg_per_batch = A_proc.sum(axis=0)
    else:
        raise ValueError("Unknown agg_layers value")

    # Re-apply node_mask if present to zero out padded rows/cols in final AAgg
    AAgg_per_batch = AAgg_per_batch * mask2d[None, :, :]

    graphs = None
    if return_nx:
        graphs = []
        for b in range(B):
            G = nx.DiGraph()
            for i in range(N):
                if node_mask is None or node_mask[i]:
                    G.add_node(i)
            M = AAgg_per_batch[b]
            for i in range(N):
                if node_mask is not None and not node_mask[i]:
                    continue
                for j in range(N):
                    if node_mask is not None and not node_mask[j]:
                        continue
                    w = float(M[i, j])
                    if w > 0:
                        G.add_edge(j, i, weight=w)  # edge from source j -> target i (consistent with A[query, key])
            graphs.append(G)

    return AAgg_per_batch, graphs

Plotting helpers

In [8]:
def plot_attention_graph_heatmap(
    AAgg_per_batch: np.ndarray,
    out_dir: str,
    cmap: str = "magma",
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    prefix: str = "attngraph"
) -> List[str]:
    """
    Save heatmaps per batch. AAgg_per_batch: (B, N, N)
    Returns list of filenames.
    """
    os.makedirs(out_dir, exist_ok=True)
    B, N, _ = AAgg_per_batch.shape
    saved = []
    if vmin is None:
        vmin = float(AAgg_per_batch.min())
    if vmax is None:
        vmax = float(AAgg_per_batch.max())

    for b in range(B):
        fig, ax = plt.subplots(figsize=(6, 5))
        sns.heatmap(AAgg_per_batch[b], cmap=cmap, vmin=vmin, vmax=vmax, ax=ax)
        ax.set_title(f"{prefix} - batch {b}")
        ax.set_xlabel("Source node index (j)")
        ax.set_ylabel("Target node index (i)")
        plt.tight_layout()
        fn = os.path.join(out_dir, f"{prefix}_batch{b}.png")
        fig.savefig(fn, dpi=150, bbox_inches="tight")
        plt.close(fig)
        saved.append(fn)
    return saved

def overlay_attention_on_graph(
    G: Optional[nx.Graph],
    AAgg: np.ndarray,
    node_positions: Optional[Dict[int, Tuple[float,float]]] = None,
    out_path: Optional[str] = None,
    top_k_edges: int = 50,
    cmap: str = "coolwarm",
    node_size: int = 300
) -> str:
    """
    Draw strongest attention edges (from AAgg) on the provided graph topology.
    - G: original graph (networkx) or None (we'll create nodes)
    - AAgg: (N, N) matrix where entry (i,j) = influence from j -> i
    - node_positions: dict node->(x,y). If None, use spring_layout
    - top_k_edges: number of edges to display (by weight)
    Returns path to saved figure (if out_path provided) or temp filename.
    """
    N = AAgg.shape[0]
    if G is None:
        Gplot = nx.DiGraph()
        Gplot.add_nodes_from(range(N))
    else:
        Gplot = G.copy()
    # Build list of edges with weights
    edges = []
    for i in range(N):
        for j in range(N):
            w = float(AAgg[i, j])
            if w > 0:
                edges.append((j, i, w))  # j -> i
    # take top-k edges by weight
    edges = sorted(edges, key=lambda x: -x[2])[:top_k_edges]
    H = nx.DiGraph()
    H.add_nodes_from(Gplot.nodes())
    for u,v,w in edges:
        H.add_edge(u, v, weight=w)
    if node_positions is None:
        pos = nx.spring_layout(Gplot, seed=42)
    else:
        pos = node_positions
    plt.figure(figsize=(8, 6))
    # draw base nodes
    nx.draw_networkx_nodes(Gplot, pos, node_size=node_size)
    # draw base edges faintly
    try:
        nx.draw_networkx_edges(Gplot, pos, alpha=0.15)
    except Exception:
        pass
    # draw top attention edges with widths proportional to weight
    weights = [H[u][v]['weight'] for u,v in H.edges()]
    widths = [max(0.5, 6.0*w) for w in weights]
    nx.draw_networkx_edges(H, pos, edgelist=H.edges(), width=widths, arrowsize=12, arrowstyle='-|>')
    nx.draw_networkx_labels(Gplot, pos, font_size=8)
    plt.axis('off')
    if out_path is None:
        out_path = "attn_overlay.png"
    plt.title("Top attention edges overlay")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close()
    return out_path

In [ ]:
AAgg_per_batch, graphs = build_attention_graphs(
    data,
    agg_heads='mean',
    agg_layers='multiply',   # 'multiply' gives multi-hop aggregated attention
    residual=True,
    top_k=16,                # keep 16 outgoing edges per node (optional, saves noise)
    threshold=1e-4,
    node_mask=None,     # optional boolean mask length N
    return_nx=True,
)

# Save heatmaps
plot_attention_graph_heatmap(AAgg_per_batch, out_dir="../../maps/attngraphs")


['../maps/attngraphs/attngraph_batch0.png']

In [ ]:
# Overlay top edges on a graph layout (if you have positions)
# overlay_attention_on_graph(graphs[0], AAgg_per_batch[0], node_positions=None, out_path="overlay_batch0.png")

### Markov Chains

In [ ]:
def graph_attention_as_markov_chain(
    attn: torch.Tensor,
    out_dir: str,
    alpha: float = 0.9,           # PageRank teleportation prob
    tol: float = 1e-6,            # convergence tolerance
    max_iter: int = 500,          # power method steps
    compute_tokenrank: bool = True,
    multi_bounce_steps: int = 5,
    plot_results: bool = True,
    max_nodes_to_plot: int = 6,
):
    """
    Interpret graph attention as a discrete-time Markov chain (DTMC),
    compute multi-bounce transitions and steady-state TokenRank.

    Args:
        attn: Tensor (L, B, H, N, N) - attention weights
        out_dir: path for saving plots
        alpha: teleportation factor for PageRank
        tol: convergence threshold for steady-state
        max_iter: max iterations for power method
        compute_tokenrank: compute stationary vector (TokenRank)
        multi_bounce_steps: number of iterative transitions for multi-bounce
        plot_results: whether to visualize results
        max_nodes_to_plot: how many nodes to plot per batch
    """
    os.makedirs(out_dir, exist_ok=True)
    L, B, H, N, N2 = attn.shape
    assert N == N2, "attention maps must be square"

    attn = attn.mean(dim=2)  # average heads → (L, B, N, N)
    I = torch.eye(N, device=attn.device)

    results = {
        "tokenrank": [],
        "lambda2": [],
        "multi_bounce": [],
    }

    for b in range(B):
        for l in range(L):
            A = attn[l, b]
            # Ensure right-stochastic matrix
            A = A / (A.sum(dim=-1, keepdim=True) + 1e-8)
            # PageRank adjustment for irreducibility
            P = alpha * A + (1 - alpha) / N * torch.ones_like(A)

            # Compute λ₂ (2nd largest eigenvalue magnitude)
            eigvals = LA.eigvals(P.T)
            eigvals = eigvals.real
            eigvals_sorted = torch.sort(eigvals, descending=True).values
            lambda2 = eigvals_sorted[1].item() if len(eigvals_sorted) > 1 else 0.0

            # Power method for steady state (TokenRank)
            if compute_tokenrank:
                v = torch.ones(N, device=attn.device) / N
                for _ in range(max_iter):
                    v_next = v @ P
                    if torch.norm(v_next - v, p=2) < tol:
                        break
                    v = v_next
                tokenrank = v / v.sum()
            else:
                tokenrank = None

            # Multi-bounce attention: iterate v_{n+1} = v_n @ P
            multi_bounce = []
            v = torch.eye(N, device=attn.device)  # each node as initial state
            for k in range(multi_bounce_steps):
                v = v @ P
                multi_bounce.append(v.detach().cpu().numpy())

            results["tokenrank"].append(tokenrank.cpu().numpy() if tokenrank is not None else None)
            results["lambda2"].append(lambda2)
            results["multi_bounce"].append(multi_bounce)

            # Plot steady-state and multi-bounce results
            if plot_results:
                node_indices = range(min(N, max_nodes_to_plot))

                if tokenrank is not None:
                    plt.figure(figsize=(6, 3))
                    plt.bar(range(N), tokenrank.cpu().numpy())
                    plt.title(f"Batch {b}, Layer {l}: TokenRank (steady-state)")
                    plt.xlabel("Node index")
                    plt.ylabel("Stationary probability")
                    plt.tight_layout()
                    plt.savefig(os.path.join(out_dir, f"batch{b}_layer{l}_tokenrank.png"), dpi=150)
                    plt.close()

                for node in node_indices:
                    heat = np.stack([mb[node] for mb in multi_bounce], axis=0)
                    plt.figure(figsize=(6, 4))
                    sns.heatmap(
                        heat, cmap="viridis", cbar=True
                    )
                    plt.title(f"Batch {b}, Layer {l}, Node {node}: Multi-bounce transitions")
                    plt.xlabel("Target node")
                    plt.ylabel("Bounce step")
                    plt.tight_layout()
                    plt.savefig(os.path.join(out_dir, f"batch{b}_layer{l}_node{node}_multi_bounce.png"), dpi=150)
                    plt.close()

    return results

In [ ]:
results = graph_attention_as_markov_chain(
    attn=data.permute(0,1,4,2,3),  # (L,B,H,N,N) -> (B,L,H,N,N)
    out_dir="../../maps/dtmc_attn",
    alpha=0.9,                # teleportation prob for irreducibility
    tol=1e-6,                 # convergence tolerance
    max_iter=500,             # power method steps
    compute_tokenrank=True,   # compute steady-state TokenRank
    multi_bounce_steps=5,     # visualize 5-step transitions
    plot_results=True,        # save plots
    max_nodes_to_plot=6,      # limit visualization to first 6 nodes
)